# Từ mô phỏng đến mã Python

Mở khu vực **Máy đã tính gì ở bước này?** trên web. Dự đoán bằng lời trước khi chạy notebook. Các căn nhà dưới đây là giả lập để học cơ chế, không phải dữ liệu thị trường.

Mục tiêu: tự kiểm tra gradient, quan sát đường/mặt phẳng/xác suất, rồi đối chiếu một tệp trace JSON tải từ web. Notebook dùng cùng công thức, nhưng thực hiện lại bằng NumPy; không dùng lại mã JavaScript.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
areas=np.array([40,55,65,90,110,130.],float)
ages=np.array([30,5,20,8,35,12.],float)
prices=np.array([1.55,2.05,2.4,3.15,3.8,4.35])
def sigmoid(z):
    return np.exp(-np.logaddexp(0,-np.asarray(z)))
def train(kind='linear',alpha=.15,steps=40):
    raw=areas[:,None] if kind=='linear' else np.column_stack([areas,ages])
    y=prices if kind=='linear' else ((areas-.5*ages>72).astype(float) if kind=='logistic' else .8+.03*areas-.015*ages+np.array([.05,-.1,.08,-.06,.1,-.05]))
    mu=raw.mean(axis=0);sd=raw.std(axis=0);X=np.column_stack([np.ones(len(raw)),(raw-mu)/sd])
    theta=np.zeros(X.shape[1]);history=[]
    for step in range(steps+1):
        scores=X@theta;p=sigmoid(scores) if kind=='logistic' else scores
        loss=np.mean(np.logaddexp(0,scores)-y*scores) if kind=='logistic' else np.mean((p-y)**2)
        gradient=(1 if kind=='logistic' else 2)*X.T@(p-y)/len(y)
        history.append(dict(step=step,theta=theta.copy(),prediction=p.copy(),loss=loss,gradient=gradient.copy()))
        theta=theta-alpha*gradient
    return raw,y,X,mu,sd,history
print('Sẵn sàng: NumPy thực hiện lại các phép tính.')

## 1. Một bước cập nhật và hai cách kiểm tra gradient

Em đoán dấu của gradient cho b và w trước khi chạy. Đổi `kind` thành `multiple` hoặc `logistic` để xem thêm các hệ số. Bảng dùng hệ số cũ để tính đồng thời tất cả hệ số mới.

In [ ]:
kind='linear'
alpha=.15
raw,y,X,mu,sd,history=train(kind,alpha)
h=history[0]
theta=h['theta'];g=h['gradient']
def loss_at(t):
    z=X@t
    return np.mean(np.logaddexp(0,z)-y*z) if kind=='logistic' else np.mean((z-y)**2)
eps=1e-5
numeric=np.array([(loss_at(theta+np.eye(len(theta))[j]*eps)-loss_at(theta-np.eye(len(theta))[j]*eps))/(2*eps) for j in range(len(theta))])
assert np.allclose(g,numeric,atol=1e-6)
display(pd.DataFrame({'Cũ':theta,'Gradient giải tích':g,'Gradient số':numeric,'Mới':theta-alpha*g},index=['b']+['w'+str(j) for j in range(1,len(theta))]))
print('MSE/BCE trước:',history[0]['loss'],'; sau:',history[1]['loss'])

## 2. Cùng dữ liệu, ba loại mô hình

Đồ thị được tính từ hệ số sau 40 bước, không phải đường được vẽ sẵn. Đổi `alpha` hoặc số bước rồi chạy lại. Với logistic, đổi ngưỡng chỉ đổi nhãn; không làm đổi xác suất hay BCE.

In [ ]:
for kind in ['linear','multiple','logistic']:
    raw,y,X,mu,sd,history=train(kind)
    theta=history[-1]['theta'];weights=theta[1:]/sd;bias=theta[0]-weights@mu
    fig=plt.figure(figsize=(10,3.6))
    if kind=='multiple':
        ax=fig.add_subplot(121,projection='3d')
        a,b=np.meshgrid(np.linspace(30,140,15),np.linspace(0,40,15))
        ax.scatter(areas,ages,y,color='seagreen')
        ax.plot_surface(a,b,bias+weights[0]*a+weights[1]*b,alpha=.4)
        ax.set(xlabel='Dien tich (m2)',ylabel='Tuoi nha',zlabel='Gia (ty)')
    else:
        ax=fig.add_subplot(121)
        if kind=='linear':
            ax.scatter(areas,y,color='seagreen');ax.plot(areas,bias+weights[0]*areas)
            ax.set(xlabel='Dien tich (m2)',ylabel='Gia (ty)')
        else:
            a,b=np.meshgrid(np.linspace(30,140,60),np.linspace(0,40,50));p=sigmoid(bias+weights[0]*a+weights[1]*b)
            ax.contourf(a,b,p,levels=np.linspace(0,1,11),cmap='RdBu_r',alpha=.5)
            ax.scatter(areas,ages,c=y,cmap='RdBu_r',edgecolors='black')
            ax.contour(a,b,p,levels=[.5],colors='black')
            ax.set(xlabel='Dien tich (m2)',ylabel='Tuoi nha')
    ax.set_title(kind)
    ax2=fig.add_subplot(122);ax2.plot([h['step'] for h in history],[h['loss'] for h in history]);ax2.set(xlabel='Cap nhat',ylabel='BCE' if kind=='logistic' else 'MSE')
    fig.tight_layout();plt.show()

## 3. K-means: gán cụm trước, dời tâm sau

Đường nối và màu dùng nhãn vừa gán. Tâm mới là trung bình của các điểm trong mỗi cụm. So với nút “Tiến một phép tính” trên web. Nếu một cụm rỗng, ví dụ này giữ nguyên tâm của cụm đó.

In [ ]:
points=np.array([[[-2.2,0,2.2][i%3]+.45*np.cos(i*1.7),[-1,1.7,-.8][i%3]+.4*np.sin(i*1.7)] for i in range(18)])
centers=points[:3].copy();distances=np.linalg.norm(points[:,None,:]-centers[None,:,:],axis=2);labels=distances.argmin(axis=1)
new=np.array([points[labels==j].mean(axis=0) if np.any(labels==j) else centers[j] for j in range(3)])
old_loss=np.sum((points-centers[labels])**2);new_loss=np.sum((points-new[labels])**2)
assert new_loss<=old_loss+1e-10
fig,axs=plt.subplots(1,2,figsize=(10,3.5))
for ax,cs,title in zip(axs,[centers,new],['Gan vao tam gan nhat','Doi tam ve trung binh']):
    ax.scatter(points[:,0],points[:,1],c=labels,cmap='viridis');ax.scatter(cs[:,0],cs[:,1],marker='x',s=170,c='red')
    for point,label in zip(points,labels):ax.plot([point[0],cs[label,0]],[point[1],cs[label,1]],color='gray',alpha=.3)
    ax.set_title(title)
plt.show();print('Inertia truoc/sau:',old_loss,new_loss)
display(pd.DataFrame(distances,columns=['d1','d2','d3']).assign(cluster=labels+1))

## 4. KNN: đo khoảng cách, sắp xếp, chọn k điểm, bỏ phiếu

Em đổi `query`, `k`, `stretch` và `scaled`. Khi đổi thang đo 100 lần, chuẩn hóa từ tập đã biết phải triệt tiêu việc đổi đơn vị. KNN ở đây không có vòng lặp giảm loss.

In [ ]:
points=np.array([[-2,0],[-1.4,1.8],[-.8,-1],[.2,2.2],[.4,.1],[1,1.6],[1.6,-.7],[2.1,1.1]])
labels=np.array([0,0,0,0,1,1,1,1]);query=np.array([.1,.8]);k=3;stretch=1;scaled=False
unit=np.array([stretch,1.]);p=points*unit;q=query*unit
if scaled:
    mu=p.mean(axis=0);sd=p.std(axis=0);p=(p-mu)/sd;q=(q-mu)/sd
d=np.linalg.norm(p-q,axis=1);order=np.argsort(d,kind='stable');chosen=order[:k];votes=np.bincount(labels[chosen],minlength=2)
display(pd.DataFrame({'point':order+1,'distance':d[order],'label':labels[order],'chosen':np.isin(order,chosen)}))
plt.scatter(points[:,0],points[:,1],c=labels,cmap='coolwarm');plt.scatter(*query,marker='*',s=180,c='orange')
for i in chosen:plt.plot([query[0],points[i,0]],[query[1],points[i,1]],color='orange')
plt.show();print('Phieu:',votes,'; Du doan:',int(np.argmax(votes)))

## 5. Đối chiếu trace tải từ web bằng NumPy

Trên web, bấm **Tải dữ liệu và các bước đã xem** ở một mô phỏng hồi quy. Đưa tệp JSON vào cùng thư mục notebook, sửa `TRACE_PATH` rồi chạy ô dưới. Kiểm tra này đối chiếu lại gradient ở mỗi bước và phép cập nhật; không chỉ so con số cuối.

In [ ]:
TRACE_PATH=''  # ví dụ: 'trace-linear.json'
if not TRACE_PATH:
    print('Chưa chọn trace. Các phần trên vẫn chạy độc lập.')
else:
    trace=json.loads(Path(TRACE_PATH).read_text())
    raw=np.array([r['x'] for r in trace['samples']],float);y=np.array([r['y'] for r in trace['samples']],float)
    mu=raw.mean(axis=0);sd=raw.std(axis=0);sd[sd==0]=1
    X=np.column_stack([np.ones(len(y)),(raw-mu)/sd]);kind=trace['algorithm']
    assert np.allclose(mu,trace['scaler']['mean']) and np.allclose(sd,trace['scaler']['scale'])
    for frame in trace['frames']:
        before=np.array(frame['before']);used=np.array(frame['used']);z=X@before;p=sigmoid(z) if kind=='logistic' else z
        g=(1 if kind=='logistic' else 2)*X[used].T@(p[used]-y[used])/len(used)
        assert np.allclose(g,frame['gradient'],atol=1e-8)
        expected=before+trace['direction']*trace['rate']*g if frame['phase']=='update' and not frame['stopped'] else before
        assert np.allclose(expected,frame['theta'],atol=1e-8)
    print('Đã kiểm tra độc lập',len(trace['frames']),'phép tính, tất cả khớp NumPy.')

## Bài thực hành và nhật ký

- A: Đoán dấu gradient của nhà cuối trước và sau khi tăng giá nó 3 tỷ. Giải thích bằng sai số có dấu.
- B: Tải trace hồi quy từ web ở batch=1 hoặc batch=3. Chạy kiểm tra NumPy và giải thích mẫu nào góp gradient bằng 0.
- C: Thay tâm khởi tạo K-means hoặc đổi đơn vị của KNN. Ghi dự đoán trước, quan sát sau và giải thích sự khác nhau.

Nhật ký: Em đã đổi gì? Mô hình đổi ở phép nào? Bằng chứng nào hỗ trợ lời giải thích?